<h2>Single Value Decomposition(Pre-req)</h2>

In [1]:
import torch
import numpy as np
_ =torch.manual_seed(0)

In [2]:
d,k=10,10
w_rank=2#even though 10 columns,only 2 combinations are independent
w=torch.randn(d,w_rank) @ torch.randn(w_rank,k)
print(w)

tensor([[-1.0797,  0.5545,  0.8058, -0.7140, -0.1518,  1.0773,  2.3690,  0.8486,
         -1.1825, -3.2632],
        [-0.3303,  0.2283,  0.4145, -0.1924, -0.0215,  0.3276,  0.7926,  0.2233,
         -0.3422, -0.9614],
        [-0.5256,  0.9864,  2.4447, -0.0290,  0.2305,  0.5000,  1.9831, -0.0311,
         -0.3369, -1.1376],
        [ 0.7900, -1.1336, -2.6746,  0.1988, -0.1982, -0.7634, -2.5763, -0.1696,
          0.6227,  1.9294],
        [ 0.1258,  0.1458,  0.5090,  0.1768,  0.1071, -0.1327, -0.0323, -0.2294,
          0.2079,  0.5128],
        [ 0.7697,  0.0050,  0.5725,  0.6870,  0.2783, -0.7818, -1.2253, -0.8533,
          0.9765,  2.5786],
        [ 1.4157, -0.7814, -1.2121,  0.9120,  0.1760, -1.4108, -3.1692, -1.0791,
          1.5325,  4.2447],
        [-0.0119,  0.6050,  1.7245,  0.2584,  0.2528, -0.0086,  0.7198, -0.3620,
          0.1865,  0.3410],
        [ 1.0485, -0.6394, -1.0715,  0.6485,  0.1046, -1.0427, -2.4174, -0.7615,
          1.1147,  3.1054],
        [ 0.9088,  

In [3]:
U,S,V=torch.svd(w)
U_r=U[:,:w_rank]#(10,2)
V_r=V[:,:w_rank].t()
S_r=torch.diag(S[:w_rank])
B=U_r @ S_r
A=V_r
print(f'shape of B: {B.shape}, shape of A: {A.shape}')

shape of B: torch.Size([10, 2]), shape of A: torch.Size([2, 10])


In [4]:
bias=torch.randn(d)
x=torch.randn(d)
#compute y=w @ x + bias
y=w@x+bias
#compute y' using LoRa decomposition
y_prime=(B@A)@x + bias
print("original y:",y)
print("LoRa y_prime:",y_prime)

original y: tensor([ 7.2684e+00,  2.3162e+00,  7.7151e+00, -1.0446e+01, -8.1639e-03,
        -3.7270e+00, -1.1146e+01,  2.0207e+00, -9.6258e+00, -4.1163e+00])
LoRa y_prime: tensor([ 7.2684e+00,  2.3162e+00,  7.7151e+00, -1.0446e+01, -8.1638e-03,
        -3.7270e+00, -1.1146e+01,  2.0207e+00, -9.6258e+00, -4.1163e+00])


In [5]:
print("total parameters in original w:",d*k)
print("total parameters in LoRa B and A:",B.numel()+A.numel())

total parameters in original w: 100
total parameters in LoRa B and A: 40


<h1>LoRA Implementation with pytorch</h1>

In [6]:
import torch  # Core PyTorch library for tensor operations and neural networks
import torch.nn as nn  # Neural network modules (layers, activations, etc.)
import torchvision.transforms as transforms  # Image preprocessing and augmentation utilities
import torchvision.datasets as datasets  # Built-in dataset loaders (MNIST, CIFAR, ImageNet, etc.)
from tqdm import tqdm  # Progress bar for training loops
import matplotlib.pyplot as plt  # Data visualization and plotting

In [7]:
#make torch deterministic
_=torch.manual_seed(0)

we will be training a network to classify MINIST digits and then fine-tune the network on a particular in which it didnt perform that well using LoRA

In [ ]:
#transform images to tensor(matrix) and normalize
transform=transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,),(0.3081,))])

#load MNIST dataset
minst_trainset=datasets.MNIST(root='./data',train=True,download=True,transform=transform)

#create a data loader for training(for batching,shuffling,etc)
train_loader=torch.utils.data.DataLoader(minst_trainset,batch_size=10,shuffle=True)

#load MNIST test dataset
minst_testset=datasets.MNIST(root='./data',train=False,download=True,transform=transform)
test_loader=torch.utils.data.DataLoader(minst_testset,batch_size=10,shuffle=False)

#define the device
device=torch.device('cuda' if torch.cuda.is_available() else'cpu')


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 486kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.44MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.49MB/s]


Create a Neural Network to classify digits, making it overly complicated so that is better to show the power of LoRA

In [9]:
class MinistModel(nn.Module):
    def __init__(self,hidden_size=1000,hidden_size2=2000):# Constructor with layer sizes
        super(MinistModel,self).__init__()
        self.linear1=nn.Linear(28*28,hidden_size)# Layer 1: 784 inputs → 1000 outputs
        self.linear2=nn.Linear(hidden_size,hidden_size2)# Layer 2: 1000 → 2000
        self.linear3=nn.Linear(hidden_size2,10)#Layer 3: 2000 → 10 (digit classes 0-9)
        self.relu=nn.ReLU()
    def forward(self,img):
        x=img.view(-1,28*28)# Flatten 28×28 image to 784 numbers
        x=self.relu(self.linear1(x))# Pass through layer1, apply ReLU
        x=self.relu(self.linear2(x))# Pass through layer2, apply ReLU
        x=self.linear3(x)
        return x
model=MinistModel().to(device)

train the network for only 1 epoch to simulate pretraining of the data

In [10]:
def train(train_loader,model,epochs=5, total_iterations_limit=None):
    cross_entropy_loss=nn.CrossEntropyLoss()# Loss function for multi-class classification
    optimizer=torch.optim.Adam(model.parameters(),lr=0.001)# Adam optimizer for training

    total_iterations=0
    for epoch in range(epochs):
        model.train() # Set model to training mode
        loss_sum=0
        num_iterations=0
        data_iterator=tqdm(train_loader,desc=f"Epoch {epoch+1}/{epochs}")
        if total_iterations_limit is not None:
            data_iterator.total=total_iterations_limit
        for data in data_iterator:
            num_iterations+=1
            total_iterations+=1
            images,labels=data
            images=images.to(device)
            labels=labels.to(device)
            optimizer.zero_grad()# Clear previous gradients
            outputs=model(images.view(-1,28*28))# Forward pass
            loss=cross_entropy_loss(outputs,labels)# Compute loss
            loss_sum+=loss.item()
            avg_loss=loss_sum/num_iterations
            data_iterator.set_postfix({'loss':avg_loss})
            loss.backward()# Backpropagation
            optimizer.step()# Update model parameters
            if total_iterations_limit is not None and total_iterations>=total_iterations_limit:
                return
train(train_loader,model,epochs=1)



Epoch 1/1: 100%|██████████| 6000/6000 [07:23<00:00, 13.54it/s, loss=0.236]


keep a copy of the original weights so later we can prove that a fine-tuning with LoRA doesnt alter the original weights

In [11]:
original_weights={}
for name,param in model.named_parameters():
        original_weights[name]=param.clone().detach()

The performance of the pretrained network is shown on the test set. It performed poorly on the digit 9. Use LoRA to finetune it

In [12]:
def test():
    correct=0
    total=0
    wrong_counts=[0 for i in range(10)]  # Initialize wrong counts for each class
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            images=images.to(device)
            labels=labels.to(device)
            output=model(images.view(-1,28*28))
            for idx,i in enumerate(output):
                if torch.argmax(i)==labels[idx]:
                    correct+=1
                else:
                    wrong_counts[labels[idx]]+=1  # Increment wrong count for the true class
                total+=1
        print(f'Accuracy: {100*correct/total}%')
        for i in range(10):
            print(f'Class {i} wrong predictions: {wrong_counts[i]}')
test()

Accuracy: 95.39%
Class 0 wrong predictions: 31
Class 1 wrong predictions: 17
Class 2 wrong predictions: 46
Class 3 wrong predictions: 74
Class 4 wrong predictions: 29
Class 5 wrong predictions: 7
Class 6 wrong predictions: 36
Class 7 wrong predictions: 80
Class 8 wrong predictions: 25
Class 9 wrong predictions: 116


Let's visualize how many parameters are in the original network, before introducing the LoRA matrices

In [13]:
#print the size of the weights matrices of the network
#save the count of the total number of parameters
total_parameters_original=0
for index,layer in enumerate([model.linear1,model.linear2,model.linear3]):
    total_parameters_original+=layer.weight.nelement()+layer.bias.nelement()
    print(f'Layer{index+1}: W:{layer.weight.shape}+ B:{layer.bias.shape}:')
print(f'Total number of parameters in original model: {total_parameters_original:,}')

Layer1: W:torch.Size([1000, 784])+ B:torch.Size([1000]):
Layer2: W:torch.Size([2000, 1000])+ B:torch.Size([2000]):
Layer3: W:torch.Size([10, 2000])+ B:torch.Size([10]):
Total number of parameters in original model: 2,807,010


Define the LoRA parameterization as described in the paper.

In [14]:
class LoRAParametrization(nn.Module):
    def __init__(self,features_in,features_out,rank=1,alpha=1,device='cpu'):
        super().__init__()
        self.lora_A=nn.Parameter(torch.zeros((rank,features_out),device=device))
        self.lora_B=nn.Parameter(torch.zeros((features_in,rank),device=device))
        nn.init.normal_(self.lora_A,mean=0,std=1)

        #scale Wx by alpha/rank, where alpha is a scaling factor
        #when optimizing with adam, tuning  alpha is roughly equivalent to tuning learning rate
        #high alpha stronger LoRA effect
        self.scaling=alpha/rank
        self.enabled=True

    def forward(self,original_weights):
        #this function runs everytime the layer is asked for its weights
        if self.enabled:
            #torch.matmul computes matrix multiplication
            return original_weights + torch.matmul(self.lora_B, self.lora_A).view(original_weights.shape) * self.scaling
        else:
            return original_weights


In [15]:
import torch.nn.utils.parametrize as parametrize
def linear_layer_parametrization(layer,device,rank=1,lora_alpha=1):
    #only parametrize the weight matrix,not the bias
    features_in,features_out=layer.weight.shape
    return LoRAParametrization(features_in,features_out,rank=rank,alpha=lora_alpha,device=device)

#Normally, 'model.linear1.weight' is a static Tensor (a box of numbers).
# 'register_parametrization' deletes that box and replaces it with a FUNCTION (the LoRA forward pass).
# Now, whenever the model asks for "model.linear1.weight", PyTorch runs the LoRA calculation 
# on the fly to generate the matrix.
parametrize.register_parametrization(
    model.linear1,'weight',linear_layer_parametrization(model.linear1,device)
)
parametrize.register_parametrization(
    model.linear2,'weight',linear_layer_parametrization(model.linear2,device)
)
parametrize.register_parametrization(
    model.linear3,'weight',linear_layer_parametrization(model.linear3,device)
)

def enable_disable_lora(model,enable=True):
    for layer in [model.linear1,model.linear2,model.linear3]:
        layer.parametrizations['weight'][0].enabled=enable

display the number of parameters added by LoRA

In [16]:
total_parameters_lora=0
total_parameters_nonlora=0
for index,layer in enumerate([model.linear1,model.linear2,model.linear3]):
    total_parameters_lora+=layer.parametrizations["weight"][0].lora_A.numel()+layer.parametrizations["weight"][0].lora_B.numel()
    total_parameters_nonlora+=layer.weight.nelement()+layer.bias.nelement()
    print(f'Layer{index+1} : W:{layer.weight.shape}+ B:{layer.bias.shape} + Lora_A:{layer.parametrizations["weight"][0].lora_A.shape}+ Lora_B:{layer.parametrizations["weight"][0].lora_B.shape}')
assert total_parameters_nonlora==total_parameters_original
print(f'Total number of parameters original: {total_parameters_nonlora:,}')
print(f'Total number of parameters origina+LoRa: {total_parameters_lora+total_parameters_nonlora:,}')




Layer1 : W:torch.Size([1000, 784])+ B:torch.Size([1000]) + Lora_A:torch.Size([1, 784])+ Lora_B:torch.Size([1000, 1])
Layer2 : W:torch.Size([2000, 1000])+ B:torch.Size([2000]) + Lora_A:torch.Size([1, 1000])+ Lora_B:torch.Size([2000, 1])
Layer3 : W:torch.Size([10, 2000])+ B:torch.Size([10]) + Lora_A:torch.Size([1, 2000])+ Lora_B:torch.Size([10, 1])
Total number of parameters original: 2,807,010
Total number of parameters origina+LoRa: 2,813,804


In [17]:
for name,param in model.named_parameters():
    if 'lora' not in name:
        print(f'freezing non-LoRa parameter: {name}')
        param.requires_grad=False

mnist_trainset=datasets.MNIST(root='./data',train=True,download=True,transform=transform)
#This creates a list of True/False values. True if the digit is a 9, False otherwise.
exclude_indices=mnist_trainset.targets==9

# We replace the dataset with ONLY the images where the mask was True.
mnist_trainset.data=mnist_trainset.data[exclude_indices]
mnist_trainset.targets=mnist_trainset.targets[exclude_indices]
train_loader=torch.utils.data.DataLoader(mnist_trainset,batch_size=10,shuffle=True)
train(train_loader,model,epochs=1,total_iterations_limit=100)

freezing non-LoRa parameter: linear1.bias
freezing non-LoRa parameter: linear1.parametrizations.weight.original
freezing non-LoRa parameter: linear2.bias
freezing non-LoRa parameter: linear2.parametrizations.weight.original
freezing non-LoRa parameter: linear3.bias
freezing non-LoRa parameter: linear3.parametrizations.weight.original


Epoch 1/1:  99%|█████████▉| 99/100 [00:03<00:00, 30.61it/s, loss=0.124]


Test the network with LoRA enabled, digit 9 should be classified better

In [19]:
enable_disable_lora(model,enable=True)
test()

Accuracy: 89.7%
Class 0 wrong predictions: 93
Class 1 wrong predictions: 37
Class 2 wrong predictions: 71
Class 3 wrong predictions: 226
Class 4 wrong predictions: 143
Class 5 wrong predictions: 106
Class 6 wrong predictions: 49
Class 7 wrong predictions: 181
Class 8 wrong predictions: 117
Class 9 wrong predictions: 7
